In [8]:
# 1. 필요한 라이브러리
from bs4 import BeautifulSoup
import pandas as pd
import re

from urllib.robotparser import RobotFileParser

- robots.txt: /best-books 수집 가능(True), Crawl-delay 1초

In [3]:
# 2. BEST_STATIC_HTML, BEST_RENDERED_HTML 정의
# 네가 지금 보낸 긴 코드 실행

In [9]:
# 종합 실습용 페이지 — 두 버전 HTML(네트워크 불필요). 구조가 Part 1~5와 다릅니다(새 셀렉터!).
BEST_STATIC_HTML = """<!DOCTYPE html><html lang="ko"><head><title>모두마켓 베스트 도서</title></head>
<body><h1>이주의 베스트 도서</h1>
  <ul id="book-list"><li class="loading">불러오는 중...</li></ul>
  <script src="/static/render_books.js"></script>
</body></html>"""

BEST_RENDERED_HTML = """<!DOCTYPE html><html lang="ko"><head><title>모두마켓 베스트 도서</title></head>
<body><h1>이주의 베스트 도서</h1>
  <ul id="book-list">
    <li class="book" data-isbn="978-1">
      <h3 class="title">파이썬으로 시작하는 데이터 분석</h3>
      <span class="author">김데이터</span><span class="price">28,000원</span><span class="rating">4.8</span>
    </li>
    <li class="book" data-isbn="978-2">
      <h3 class="title">  모두를 위한 통계학  </h3>
      <span class="author">이통계</span><span class="price">32,000원</span><span class="rating">4.6</span>
    </li>
    <li class="book" data-isbn="978-3">
      <h3 class="title">웹 스크래핑 실전</h3>
      <span class="author">박수집</span><span class="price">₩26000</span>
    </li>
    <li class="book" data-isbn="978-4">
      <h3 class="title">머신러닝 입문</h3>
      <span class="author">최모델</span><span class="price">35000</span><span class="rating">4.9</span>
    </li>
    <li class="book" data-isbn="978-5">
      <h3 class="title">SQL 첫걸음</h3>
      <span class="author">정쿼리</span><span class="price">24,000원</span><span class="rating">4.5</span>
    </li>
  </ul>
  <script src="/static/render_books.js"></script>
</body></html>"""

print("종합 실습용 HTML 준비 완료 (정적/렌더링 2종)")

종합 실습용 HTML 준비 완료 (정적/렌더링 2종)


In [10]:
# 시나리오 1 — 정적 vs 렌더링 비교로 동적 페이지임을 확인
soup_static_b = BeautifulSoup(BEST_STATIC_HTML, "html.parser")
soup_rendered_b = BeautifulSoup(BEST_RENDERED_HTML, "html.parser")

print("정적 HTML의 .book 수  :", len(soup_static_b.select(".book")))      # 0 → 동적!
print("렌더링 HTML의 .book 수:", len(soup_rendered_b.select(".book")))   # 5
print("\n판별 결과: 소스엔 없고 화면엔 있으므로 → 동적 페이지 (Playwright 필요)")

정적 HTML의 .book 수  : 0
렌더링 HTML의 .book 수: 5

판별 결과: 소스엔 없고 화면엔 있으므로 → 동적 페이지 (Playwright 필요)


In [11]:
# 시나리오 2 — (가정) AI가 준 초안: 셀렉터가 틀렸다
def ai_book_draft(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for li in soup.select("li.item"):          # ← 틀림: 실제는 li.book
        rows.append({"title": li.select_one(".title").get_text()})
    return pd.DataFrame(rows)

print("[검증] AI 초안 결과 행 수:", len(ai_book_draft(BEST_RENDERED_HTML)))   # 0 → 셀렉터 의심!

# 수정: 실제 HTML을 보고 li.book / 올바른 셀렉터로 고친다
def book_scraper(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for li in soup.select("li.book"):
        rating_el = li.select_one(".rating")           # 없을 수 있음(결측!)
        rows.append({
            "isbn": li.get("data-isbn"),
            "title": li.select_one(".title").get_text(),
            "author": li.select_one(".author").get_text(strip=True),
            "price_raw": li.select_one(".price").get_text(strip=True),
            "rating": rating_el.get_text(strip=True) if rating_el else None,
        })
    return pd.DataFrame(rows)

books = book_scraper(BEST_RENDERED_HTML)
print("[수정 후] 행 수:", len(books))
display(books)

[검증] AI 초안 결과 행 수: 0
[수정 후] 행 수: 5


,isbn,title,author,price_raw,rating
0,978-1,파이썬으로 시작하는 데이터 분석,김데이터,"28,000원",4.8
1,978-2,모두를 위한 통계학,이통계,"32,000원",4.6
2,978-3,웹 스크래핑 실전,박수집,₩26000,NaN
3,978-4,머신러닝 입문,최모델,35000,4.9
4,978-5,SQL 첫걸음,정쿼리,"24,000원",4.5


In [12]:
# 시나리오 3 — 정제(공백·결측·가격) + robots.txt 검토
books_clean = books.copy()
books_clean["title"] = books_clean["title"].str.strip()                 # 공백 제거(D+005)
books_clean["rating"] = books_clean["rating"].fillna("평점없음")          # 결측 대체(D+003)
books_clean["price"] = books_clean["price_raw"].str.replace(r"[^0-9]", "", regex=True).astype(int)  # 숫자만(D+005)
books_final = books_clean[["isbn", "title", "author", "price", "rating"]]

print("[정제 완료]")
display(books_final)

# robots.txt 윤리 검토(로컬 시연)
rp_b = RobotFileParser()
rp_b.parse(["User-agent: *", "Allow: /", "Disallow: /admin/", "Crawl-delay: 1"])
print("이 페이지(/best-books) 수집 가능? →", rp_b.can_fetch("*", "/best-books"))
print("평균 도서 가격:", int(books_final["price"].mean()), "원")

[정제 완료]


,isbn,title,author,price,rating
0,978-1,파이썬으로 시작하는 데이터 분석,김데이터,28000,4.8
1,978-2,모두를 위한 통계학,이통계,32000,4.6
2,978-3,웹 스크래핑 실전,박수집,26000,평점없음
3,978-4,머신러닝 입문,최모델,35000,4.9
4,978-5,SQL 첫걸음,정쿼리,24000,4.5


이 페이지(/best-books) 수집 가능? → True
평균 도서 가격: 29000 원


In [13]:
# 종합 — 정리된 데이터로 간단 분석: 평점 있는 도서를 평점 높은 순으로
rated_books = books_final[books_final["rating"] != "평점없음"].copy()
rated_books["rating_num"] = rated_books["rating"].astype(float)
top_books = rated_books.sort_values("rating_num", ascending=False)
print("평점 높은 순 베스트 도서:")
display(top_books[["title", "author", "price", "rating"]])

평점 높은 순 베스트 도서:


,title,author,price,rating
3,머신러닝 입문,최모델,35000,4.9
0,파이썬으로 시작하는 데이터 분석,김데이터,28000,4.8
1,모두를 위한 통계학,이통계,32000,4.6
4,SQL 첫걸음,정쿼리,24000,4.5


In [14]:

rated = books_final[books_final["rating"] != "평점없음"]

print("평점이 있는 도서 수:", len(rated))
print("평점 있는 도서 평균 가격:", int(rated["price"].mean()), "원")
display(rated[["title", "price", "rating"]])

평점이 있는 도서 수: 4
평점 있는 도서 평균 가격: 29750 원


,title,price,rating
0,파이썬으로 시작하는 데이터 분석,28000,4.8
1,모두를 위한 통계학,32000,4.6
3,머신러닝 입문,35000,4.9
4,SQL 첫걸음,24000,4.5


# 모두마켓 베스트 도서 — 수집 리포트

## 1. 페이지 판별
- 정적/동적: **동적**
  (근거: 소스의 .book 수 0개, 화면의 .book 수 5개)
- 선택한 도구: **Playwright + BeautifulSoup**
  (이유: JavaScript가 실행된 후에 .book 데이터가 생성되기 때문)

## 2. 수집 (AI 보조 4단계)
- 목적: 각 도서의 제목·저자·가격·평점을 표로 수집
- AI 초안 검증 결과: **행 수 0개** → 문제: **셀렉터(.item)가 실제 HTML(.book)과 달랐음**
- 수정: **li.item → li.book** 등으로 수정하여 **최종 5행 수집**

## 3. 정제 (재적용한 랭글링)
- 공백 제거: **title**
- 결측 대체: **rating(평점없음)**
- 가격 숫자화: **price (28,000원 → 28000)**

## 4. 윤리 검토
- robots.txt: **/best-books 수집 가능(True), Crawl-delay 1초**
- 요청 예의: **time.sleep()과 User-Agent를 적용하여 과도한 요청을 피한다.**
- 저작권/개인정보: **개인정보는 포함되지 않았으며, 수집 데이터는 학습 목적에만 사용하고 재배포하지 않는다.**

## 5. 다음 단계
- **가격과 평점의 관계를 분석하거나 평균 가격, 최고·최저 가격 등을 분석할 수 있다.**